In [22]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import random
import sacrebleu
import Levenshtein
import javalang

import re
import bertviz

from models import CodeTokenizer, CodeSet, CodeSetTransformer, VanillaSeq2Seq, AttendedSeq2Seq, Transformer, device

In [23]:

# Setting a manual seed for reproducability (useful for ablations)
torch.manual_seed(67)

def seed_worker(worker_id): # Manual seeding init function for dataloaders
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)
g = torch.Generator()
g.manual_seed(42)


def test(model, tokenizer, test_dataloader):
    model.eval()
    s_matches = 0
    s_editd = 0
    s_bleu = 0
    s_syntax = 0
    i=1

    is_transformer = isinstance(model, Transformer)

    with torch.no_grad():
        for batch in test_dataloader:
            if is_transformer:
                buggy, fixed_inf, fixed_tra = batch
                buggycode = tokenizer.detokenize(buggy.cpu().tolist())
                fixedcode = tokenizer.detokenize(fixed_tra.cpu().tolist())
                pred, _, __, ___ = model(buggy)
                # print(pred.shape)
                predcode = tokenizer.detokenize(pred.argmax(-1).cpu().tolist())
            else:
                buggy, fixed = batch
                buggycode = tokenizer.detokenize(buggy.cpu().tolist())
                fixedcode = tokenizer.detokenize(fixed.cpu().tolist())
                pred = model(buggy)
                # print(pred.shape)
                predcode = tokenizer.detokenize(pred.argmax(-1).cpu().tolist())
            
            for bug,fix,pre in zip(buggycode,fixedcode,predcode):
                match_score = sum([1 if i==j else 0 for i,j in zip(fix,pre)])
                edit_distance = Levenshtein.distance(fix,pre)
                bleu = sacrebleu.sentence_bleu(pre, [fix]).score
                try:
                    wrapped = f"public class Temp {{ {pre} }}"
                    _ = javalang.parse.parse(wrapped)
                    syntax_check = "Valid"
                    s_syntax+=1
                except javalang.parser.JavaSyntaxError:  # <--- LLM used for this except branch
                    syntax_check = "Invalid"
                except:
                    syntax_check = "Valid"

                if i%100==0: # or syntax_check=="Valid"
                    print(f"{i}. " +f"\t[To Debug]>\t{bug}\n\t[Model]>\t{pre}\n\t[Actual]>\t{fix}\n")
                    print(f"----> Match_score= {match_score}\n----> Edit_distance= {edit_distance}\n----> BLEU= {round(bleu,3)}\n----> Syntax_check= {syntax_check}\n\n")
                
                s_matches+=match_score
                s_editd+=edit_distance
                s_bleu+=bleu
                i+=1

    print("avg_absolute_matches =",round(s_matches/i,3))
    print("avg_edit_distance =",round(s_editd/i,3))
    print("avg_bleu_score =",round(s_bleu/i,3))
    print("fraction_valid_syntax =",round(s_syntax/i,3))

In [24]:
# Loading our dataset! using only small code set for easier and faster computing
print("Loading datasets...", end='')
train_data = pd.read_parquet('.\\Datasets\\small\\train.parquet').to_numpy()
test_data = pd.read_parquet('.\\Datasets\\small\\test.parquet').to_numpy()
print(" Done.\n")

Loading datasets... Done.



In [25]:
max_seq_len = 150

# Tokenizer set up
print("Setting up tokenizer...", end='')
tokenizer = CodeTokenizer(max_seq_len)
tokenizer.vocabularize(train_data)
vocab_size = len(tokenizer.vocab)
print(" Done.\n")

# Select model here: "vanilla", "attended", "transformer"
selected_model = "transformer"

Setting up tokenizer... Done.



In [26]:
# Setting up dataloaders
print("Initialising dataloaders...", end='')
if selected_model == "transformer":
    DatasetClass = CodeSetTransformer
    batch_size = 64
else:
    DatasetClass = CodeSet
    batch_size = 256

test_loader = DataLoader(DatasetClass(test_data, tokenizer, sample_data=None), batch_size=batch_size, shuffle=True, num_workers=0,  worker_init_fn=seed_worker)
print(" Done.\n")

Initialising dataloaders... Done.



In [27]:
# Initialisation
print("Initialising model...", end='')
if selected_model == "vanilla":
    model = VanillaSeq2Seq(max_seq_len=max_seq_len, vocab_size=vocab_size, emb_dim=8, hidden_size=16, pad_id=tokenizer.pad_id, tf_ratio=0.5)
    model_name = "rnn_model"
elif selected_model == "attended":
    model = AttendedSeq2Seq(max_seq_len=max_seq_len, vocab_size=vocab_size, emb_dim=8, hidden_size=16, pad_id=tokenizer.pad_id, tf_ratio=0.5)
    model_name = "attended_rnn_model"
elif selected_model == "transformer":
    model = Transformer(N=3, h=4, dmodel=64, dk=16, dv=16, max_seq_len=max_seq_len, vocab_size=vocab_size, bos=tokenizer.bos, pad_id=tokenizer.pad_id, viz=True)
    model_name = "transformer_model"                                                                                        # viz=True for bertviz visualisation!
print(" Done.\n")

Initialising model... Done.



In [28]:
# Loading model with least val_loss
print("Loading model with the best parameters...", end='')
model.load_state_dict(torch.load(f'.\\Saved Models\\{model_name}_.pth'))
print(" Done.\n")

Loading model with the best parameters... Done.



In [29]:
# Testing
print("Testing... (Press ALT+C to interrupt testing)")
test(model, tokenizer, test_dataloader=test_loader)
print("\nTesting complete!.\n")

Testing... (Press ALT+C to interrupt testing)
100. 	[To Debug]>	protected void start ( TYPE_1 b ) { if ( b == null ) throw new TYPE_2 ( ) ; result = true ; }
	[Model]>	protected void start ( TYPE_1 b ) { if ( b == null ) throw new TYPE_2 ( ) ; }
	[Actual]>	protected boolean start ( TYPE_1 b ) { if ( b == null ) throw new java.lang.NullPointerException ( ) ; result = true ; return false ; }

----> Match_score= 14
----> Edit_distance= 65
----> BLEU= 50.56
----> Syntax_check= Valid


200. 	[To Debug]>	public void METHOD_1 ( TYPE_1 name , TYPE_2 VAR_1 ) { TYPE_3 . i ( VAR_2 , STRING_1 ) ; VAR_3 = ( ( TYPE_4 ) ( VAR_1 ) ) ; VAR_3 . METHOD_2 ( this ) ; METHOD_3 ( false ) ; }
	[Model]>	public void METHOD_1 ( TYPE_1 name , TYPE_2 VAR_1 ) { TYPE_3 VAR_2 = VAR_3 ; VAR_3 = new TYPE_4 ( ) ; VAR_3 . METHOD_2 ( this ) ; METHOD_3 ( ) ; }
	[Actual]>	public void METHOD_1 ( TYPE_1 name , TYPE_2 VAR_1 ) { TYPE_3 . i ( VAR_2 , STRING_1 ) ; VAR_3 = ( ( TYPE_4 ) ( VAR_1 ) ) ; VAR_3 . METHOD_2 ( this ) ; MET

In [ ]:
# Demo
buggy_code = "protected void METHOD_1 ( ) { listener . METHOD_2 ( this ) ; super . METHOD_1 ( ) ; }"

input = torch.tensor([tokenizer.tokenize_single(buggy_code)], device=device)
pred, enc_attn, dec_attn, cross_attn = model(input)
pred_code = tokenizer.detokenize_single((pred.argmax(-1).cpu().tolist())[0])
print("Debugged Code (acc. to Model):\n>",pred_code)

if model_name=="transformer_model":   # LLM was extensively used for the know-how on using bertviz with transformer model
    buggy_sep = [i for i in re.split(r'([. ])', buggy_code) if i!=' '] + ['<EOS>']
    output_sep = [i for i in re.split(r'([. ])', pred_code) if i!=' '] + ['<EOS>']
    # print(buggy_sep)
    enc_attention = [layer[0:1, :, :len(buggy_sep), :len(buggy_sep)].cpu() for layer in enc_attn]
    dec_attention = [layer[0:1, :, :len(output_sep), :len(output_sep)].cpu() for layer in dec_attn]
    cross_attention = [layer[0:1, :, :len(output_sep), :len(buggy_sep)].cpu() for layer in cross_attn]

    # print(cross_attention[0].shape)
    # print(len(output_sep), len(buggy_sep))


    # print(len(buggy_sep), len(output_sep))
    # print(cross_attn[0].shape)
    bertviz.model_view(
        encoder_attention=enc_attention,
        decoder_attention=dec_attention,
        cross_attention=cross_attention,
        encoder_tokens= buggy_sep,
        decoder_tokens = output_sep
    )

    bertviz.head_view(
        cross_attention=cross_attention,
        encoder_tokens=buggy_sep,
        decoder_tokens=output_sep
    )

Debugged Code (acc. to Model):
> protected void METHOD_1 ( ) { super . METHOD_1 ( ) ; }
25 16
torch.Size([1, 4, 150, 150])


<IPython.core.display.Javascript object>